# API & MCP App Research Agent — Jupyter Notebook

This notebook creates a practical research pipeline for your Product Ops assignment.

**What it does**
1. Takes a list of apps and categories.
2. Searches the web for each app's official API/developer documentation.
3. Reads selected pages and extracts structured facts using an LLM.
4. Saves results and evidence URLs to CSV/JSON as it goes.
5. Optionally runs a second-pass verification on a sample.
6. Produces simple category/authentication/buildability summaries.

**Important:** the agent must not guess. If documentation does not confirm a field, it should return `Unknown` and flag it for manual review. Review the evidence before submitting your final report.

**Services used:** Tavily for web search and OpenAI for extraction. Composio is an integration/tool-execution platform; it is not a substitute for the web-search source. An optional Composio MCP connection example is included near the end.

## 1. Install dependencies

Run this cell once. If Jupyter asks you to restart the kernel after installation, do so and continue from the next cell.

In [ ]:
%pip install -q requests beautifulsoup4 pandas pydantic python-dotenv matplotlib

## 2. Add your Tavily API keyOllama runs the language model locally on your Mac, so no OpenAI API key or paid OpenAI credits are required. Tavily is still used for web search.

In [ ]:
import osfrom getpass import getpassTAVILY_API_KEY = getpass("Tavily API key: ").strip()os.environ["TAVILY_API_KEY"] = TAVILY_API_KEYassert TAVILY_API_KEY, "Tavily API key is required."print("Tavily API key loaded for this notebook session.")

## 3. Imports and configuration

In [ ]:
import jsonimport reimport timeimport requestsimport pandas as pdfrom bs4 import BeautifulSoupfrom pydantic import BaseModel, Fieldfrom typing import List, Optionalfrom urllib.parse import urlparsefrom IPython.display import displayOLLAMA_URL = "http://localhost:11434/api/chat"MODEL = "llama3.2:3b"TAVILY_URL = "https://api.tavily.com/search"MAX_SEARCH_RESULTS = 5MAX_PAGES_TO_READ = 3REQUEST_TIMEOUT = 30PAUSE_BETWEEN_APPS_SECONDS = 1.0def ask_local_ai(prompt: str, system_prompt: str = "You are a careful research analyst.") -> str:    """Send a prompt to the locally running Ollama model."""    response = requests.post(        OLLAMA_URL,        json={            "model": MODEL,            "messages": [                {"role": "system", "content": system_prompt},                {"role": "user", "content": prompt}            ],            "format": "json",            "stream": False,            "options": {"temperature": 0}        },        timeout=180    )    response.raise_for_status()    return response.json()["message"]["content"]print("Local Ollama research agent configuration ready.")

In [ ]:
# Test the local Ollama connection before continuing.test_response = ask_local_ai('Return JSON with fields topic and definition. Set topic to API and define it in one short sentence.')print(test_response)

## 4. Load the complete 100-app assignment listPlace the updated `apps.csv` in the same folder as this notebook. It must contain `app_name`, `category`, and preferably `website_or_hint`. The repository version already has these columns.

In [ ]:
apps_df = pd.read_csv("apps.csv")required_columns = {"app_name", "category"}missing = required_columns - set(apps_df.columns)assert not missing, f"Missing required columns in apps.csv: {missing}"apps_df = apps_df.drop_duplicates(subset=["app_name"]).reset_index(drop=True)if "website_or_hint" not in apps_df.columns:    apps_df["website_or_hint"] = ""display(apps_df.head(10))print(f"Apps queued: {len(apps_df)}")assert len(apps_df) == 100, f"Expected 100 apps, found {len(apps_df)}. Check apps.csv."

## 5. Web search tool

The agent searches for official developer/API documentation first. Search results are leads, not proof; the extraction prompt will require evidence URLs and will allow `Unknown`.

In [ ]:
def tavily_search(query: str, max_results: int = MAX_SEARCH_RESULTS) -> list:
    payload = {
        "api_key": os.environ["TAVILY_API_KEY"],
        "query": query,
        "search_depth": "advanced",
        "max_results": max_results,
        "include_answer": False,
        "include_raw_content": False,
    }
    response = requests.post(TAVILY_URL, json=payload, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    data = response.json()
    return data.get("results", [])


def search_app_docs(app_name: str) -> list:
    queries = [
        f"{app_name} official developer API documentation authentication",
        f"{app_name} official API getting started developer portal",
        f"{app_name} official MCP server Model Context Protocol",
    ]
    combined = []
    seen_urls = set()

    for query in queries:
        try:
            results = tavily_search(query)
            for item in results:
                url = item.get("url", "")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    combined.append({
                        "title": item.get("title", ""),
                        "url": url,
                        "content": item.get("content", ""),
                        "score": item.get("score", None),
                    })
        except Exception as e:
            print(f"Search warning for {app_name}: {e}")

    return combined


# Test the search tool on one app before running the full list.
test_results = search_app_docs(apps_df.iloc[0]["app_name"])
display(pd.DataFrame(test_results).head(8))

## 6. Read web pages

Some websites block automated requests or render content with JavaScript. This simple reader works for many static documentation pages; blocked/empty pages should be checked manually in a browser.

In [ ]:
def read_webpage(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; AppResearchAgent/1.0)"
    }
    try:
        response = requests.get(
            url, headers=headers, timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
            tag.decompose()

        text = soup.get_text(" ", strip=True)
        text = re.sub(r"\s+", " ", text)
        return text[:12000]
    except Exception as e:
        return f"[PAGE_READ_ERROR] {type(e).__name__}: {e}"


def collect_sources(app_name: str) -> list:
    search_results = search_app_docs(app_name)
    sources = []

    for result in search_results[:MAX_PAGES_TO_READ]:
        page_text = read_webpage(result["url"])
        sources.append({
            "title": result["title"],
            "url": result["url"],
            "search_snippet": result["content"],
            "page_text": page_text,
        })

    return sources


# Quick test
test_sources = collect_sources(apps_df.iloc[0]["app_name"])
print("Sources collected:", len(test_sources))
for s in test_sources:
    print("\nTITLE:", s["title"])
    print("URL:", s["url"])
    print("TEXT PREVIEW:", s["page_text"][:350])

## 7. Define the structured research record

These fields are the output schema. The agent must distinguish confirmed facts from missing information.

In [ ]:
class AppResearchRecord(BaseModel):
    app_name: str
    category: str
    description: str
    authentication_methods: List[str] = Field(default_factory=list)
    credential_access: str  # Self-serve, Gated, Mixed, Unknown
    api_available: str      # Yes, No, Unknown
    api_type_or_surface: str
    api_docs_url: str
    mcp_status: str          # Native MCP, Third-party/Composio integration, No evidence, Unknown
    buildability: str        # Easy, Moderate, Difficult, Unknown
    main_blocker: str
    evidence: List[str] = Field(default_factory=list)
    confidence: str          # High, Medium, Low
    needs_manual_review: bool
    notes: str = ""

## 8. Local extraction agent (Ollama)The local model extracts a structured record from collected evidence. It does not require OpenAI API credits. Unknown or weakly supported facts are flagged for manual review.

In [ ]:
SYSTEM_PROMPT = """You are a careful API and integration research analyst.Use only the supplied evidence. Do not use prior knowledge to fill gaps.Never invent API endpoints, authentication methods, MCP support, or documentation URLs.If evidence is insufficient, write Unknown and set needs_manual_review=true.Prefer official app developer documentation and official support pages.A search result snippet alone is weak evidence; use page text where available.Distinguish native MCP support from third-party connectors/integrations.Return valid JSON only, matching the requested fields."""def extract_app_record(app_name: str, category: str, sources: list) -> dict:    evidence_text = json.dumps(sources, ensure_ascii=False)    user_prompt = f"""Research this application using only the evidence below.App: {app_name}Category: {category}Return one JSON object with exactly these fields:{{  "app_name": "{app_name}",  "category": "{category}",  "description": "One concise sentence describing the product, or Unknown",  "authentication_methods": ["OAuth 2.0", "API key", "Basic auth", "JWT", "Other", "Unknown"],  "credential_access": "Self-serve, Gated, Mixed, or Unknown",  "api_available": "Yes, No, or Unknown",  "api_type_or_surface": "Brief description or Unknown",  "api_docs_url": "Official documentation URL from evidence, or Unknown",  "mcp_status": "Native MCP, Third-party/Composio integration, No evidence, or Unknown",  "buildability": "Easy, Moderate, Difficult, or Unknown",  "main_blocker": "Main practical blocker, None identified, or Unknown",  "evidence": ["URLs from supplied evidence only"],  "confidence": "High, Medium, or Low",  "needs_manual_review": true,  "notes": "Caveats and unconfirmed facts"}}Rules:- Do not treat a normal website login as API authentication.- Self-serve means a developer can obtain API credentials without contacting sales/support.- Gated means approval, sales contact, partner status, or manual review is required.- Mixed means some access is self-serve but important scopes/features require approval.- Only say Native MCP when the vendor itself documents its MCP server.- Third-party connectors do not prove native MCP support.- Buildability considers auth setup, API coverage, approval barriers, docs quality, and known limits.- Search snippets alone are not enough for high confidence.- Set needs_manual_review=true when important details are unknown or weakly supported.Evidence:{evidence_text}"""    raw = ask_local_ai(user_prompt, SYSTEM_PROMPT)    data = json.loads(raw)    validated = AppResearchRecord.model_validate(data)    return validated.model_dump()# Test extraction after the Ollama connection test succeeds.# test_record = extract_app_record(apps_df.iloc[0]["app_name"], apps_df.iloc[0]["category"], test_sources)# display(pd.Series(test_record))

## 9. Run the research agent across the app list

This loop checkpoints after every app. If the notebook stops, rerun it: already completed apps in `research_results.json` will be skipped.

Start with 5–10 apps. Inspect the records. Then run the full 100.

In [ ]:
OUTPUT_JSON = "research_results.json"OUTPUT_CSV = "research_results.csv"def load_existing_results(path=OUTPUT_JSON):    if os.path.exists(path):        with open(path, "r", encoding="utf-8") as f:            return json.load(f)    return []def save_results(records, json_path=OUTPUT_JSON, csv_path=OUTPUT_CSV):    with open(json_path, "w", encoding="utf-8") as f:        json.dump(records, f, indent=2, ensure_ascii=False)    pd.DataFrame(records).to_csv(csv_path, index=False)def run_research(apps: pd.DataFrame, limit: Optional[int] = None):    records = load_existing_results()    completed = {r.get("app_name") for r in records}    work = apps.head(limit) if limit is not None else apps    for _, row in work.iterrows():        app_name = str(row["app_name"]).strip()        category = str(row["category"]).strip()        if app_name in completed:            print(f"Skipping already completed: {app_name}")            continue        print(f"Researching: {app_name}")        try:            sources = collect_sources(app_name)            record = extract_app_record(app_name, category, sources)            record["source_count"] = len(sources)            record["research_status"] = "Needs human review" if record["needs_manual_review"] else "AI extracted - verify"            records.append(record)            save_results(records)            completed.add(app_name)            print(f"Saved: {app_name} | API: {record['api_available']} | Review: {record['needs_manual_review']}")        except Exception as e:            print(f"FAILED: {app_name}: {type(e).__name__}: {e}")            records.append({                "app_name": app_name, "category": category,                "description": "Research failed; rerun or review manually.",                "authentication_methods": [], "credential_access": "Unknown",                "api_available": "Unknown", "api_type_or_surface": "Unknown",                "api_docs_url": "Unknown", "mcp_status": "Unknown",                "buildability": "Unknown", "main_blocker": "Research pipeline error",                "evidence": [], "confidence": "Low", "needs_manual_review": True,                "notes": str(e), "source_count": 0, "research_status": "Failed - retry"            })            save_results(records)            completed.add(app_name)        time.sleep(PAUSE_BETWEEN_APPS_SECONDS)    return records# Start with five apps. Review the results before scaling up.records = run_research(apps_df, limit=5)display(pd.DataFrame(records).tail())

## 10. Run the remaining applicationsAfter reviewing the first five records and correcting any issues, run the full list. Previously saved app records are skipped. This uses Tavily search requests and local Ollama inference; Tavily usage may be subject to its free-tier limits.

In [ ]:
# Run all apps. Completed app names in research_results.json are skipped.# records = run_research(apps_df, limit=None)# results_df = pd.DataFrame(records)# display(results_df)

## 11. Independent second-pass reviewThis second pass uses fresh search evidence and the same local Ollama model. It helps flag inconsistencies, but it is not a substitute for manual ground-truth verification.

In [ ]:
def verify_app_record(record: dict) -> dict:    app_name = record["app_name"]    fresh_results = search_app_docs(f"{app_name} official API authentication developer docs MCP")    fresh_sources = []    for item in fresh_results[:MAX_PAGES_TO_READ]:        fresh_sources.append({            "title": item["title"],            "url": item["url"],            "search_snippet": item["content"],            "page_text": read_webpage(item["url"])        })    prompt = f"""Review the initial record against this independently collected evidence.Initial record:{json.dumps(record, ensure_ascii=False)}New evidence:{json.dumps(fresh_sources, ensure_ascii=False)}Return JSON with these fields:{{  "app_name": "{app_name}",  "verification_status": "Confirmed, Corrected, Contradicted, or Insufficient evidence",  "fields_to_change": ["field names"],  "corrections": {{}},  "reason": "Brief explanation",  "evidence": ["supporting URLs"],  "human_review_required": true}}Do not assume the initial record is correct. Use Unknown when evidence is insufficient."""    raw = ask_local_ai(prompt, SYSTEM_PROMPT)    return json.loads(raw)# Verify a small sample only after research records exist.# results_df = pd.DataFrame(load_existing_results())# sample_size = min(3, len(results_df))# verification_records = [verify_app_record(row.to_dict()) for _, row in results_df.head(sample_size).iterrows()]# with open("verification_results.json", "w", encoding="utf-8") as f:#     json.dump(verification_records, f, indent=2, ensure_ascii=False)# display(pd.DataFrame(verification_records))

## 12. Analyze the findings

This produces descriptive summaries from the collected records. These are observations about your dataset, not universal claims about every app in the market.

In [ ]:
results_df = pd.DataFrame(load_existing_results())

if not results_df.empty:
    print("Apps researched:", results_df["app_name"].nunique())
    print("\nAPI availability:")
    display(results_df["api_available"].value_counts(dropna=False).to_frame("count"))

    print("\nCredential access:")
    display(results_df["credential_access"].value_counts(dropna=False).to_frame("count"))

    print("\nBuildability:")
    display(results_df["buildability"].value_counts(dropna=False).to_frame("count"))

    print("\nMCP status:")
    display(results_df["mcp_status"].value_counts(dropna=False).to_frame("count"))

    print("\nBy category:")
    category_summary = results_df.groupby("category").agg(
        apps=("app_name", "nunique"),
        api_yes=("api_available", lambda s: (s == "Yes").sum()),
        self_serve=("credential_access", lambda s: (s == "Self-serve").sum()),
        needs_review=("needs_manual_review", lambda s: s.astype(str).str.lower().isin(["true", "1"]).sum()),
    ).reset_index()
    display(category_summary)
else:
    print("No research results found yet. Run the research agent first.")

## 13. Find records that need human review

These are the records you should prioritize before final submission.

In [ ]:
if not results_df.empty:
    review_df = results_df[
        results_df["needs_manual_review"].astype(str).str.lower().isin(["true", "1"])
        | results_df["confidence"].isin(["Low", "Medium"])
    ]
    display(review_df[[
        "app_name", "category", "api_available", "credential_access",
        "mcp_status", "confidence", "evidence", "notes"
    ]])
    review_df.to_csv("manual_review_queue.csv", index=False)
    print("Saved manual_review_queue.csv")

## 14. Optional: connect a Composio MCP session

Use this only if your assignment requires you to demonstrate Composio. Composio can expose connected app tools through an MCP session, but that is different from proving that an app provides its own native MCP server.

Install the current Composio SDK and configure `COMPOSIO_API_KEY` in your environment. SDK method names can change between versions, so check the current official quickstart if this cell errors.

In [ ]:
# Optional setup:
# %pip install -q composio

# import os
# from getpass import getpass
# os.environ["COMPOSIO_API_KEY"] = getpass("Composio API key: ").strip()

# Current documented session-style example:
# from composio import Composio
# composio = Composio()
# session = composio.create(user_id="app-research-notebook", mcp=True)
# print("MCP URL:", session.mcp.url)
# print("MCP headers:", session.mcp.headers)

# Do not publish MCP URLs, headers, or credentials in your GitHub repository.

## 15. Export files and download them

The key outputs are:
- `research_results.csv`
- `research_results.json`
- `verification_results.json`
- `manual_review_queue.csv`

Keep your evidence URLs in the final report so a reviewer can reproduce your findings.

In [ ]:
import os

for filename in [
    "research_results.csv",
    "research_results.json",
    "verification_results.json",
    "manual_review_queue.csv",
]:
    if os.path.exists(filename):
        print(filename, "—", os.path.getsize(filename), "bytes")

# In Google Colab only, you can use:
# from google.colab import files
# files.download("research_results.csv")

## Before submitting- Confirm all 100 apps are present and assigned to the correct category.- Run the pipeline in batches and retain checkpoint files.- Open and inspect official evidence URLs, especially authentication and credential-access claims.- Manually verify a sample (for example, 20 apps) and calculate field-level accuracy against a ground-truth sheet.- Do not report hypothetical accuracy as measured accuracy.- Distinguish native MCP support from Composio or other third-party integrations.- Document blocked pages, unknown fields, API approval barriers, rate limits, pricing, and other buildability blockers.- Never commit API keys, MCP headers, or other secrets.**Local AI:** Ollama with `llama3.2:3b`. **Web search:** Tavily. Tavily may have usage limits; local extraction does not use OpenAI API credits.